# 📖 NLP Engineer 1 — Vocabulary, Tokenization & Dataset
### Ishara Arabic Sign Language Translator | Graduation Project

**مسؤوليتك:**  
أنت بتبني الأساس اللي كل الـ NLP pipeline بتاتعته:  
الـ vocabulary، الـ tokenization، والـ PyTorch Dataset اللي هيستخدمه NLP Eng 2 في الـ training.

---
**الـ Pipeline بتاعتك:**
```
train.csv  (gloss + sentence columns)
        ↓
Arabic Text Normalization
        ↓
Build Gloss Vocab  (353 tokens + 4 special)
Build Text Vocab   (695 words  + 4 special)
        ↓
Save vocab_gloss.json  +  vocab_text.json
        ↓
IsharaNLP_Dataset  (PyTorch Dataset class)
        ↓
DataLoader  → جاهز للـ training
```


## 📦 Step 1 — Install & Import

In [ ]:
import os, json, re, collections
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

print("✅ All imports successful")


## ⚙️ Step 2 — Config

In [ ]:
# ====================================================
CSV_PATH      = "/kaggle/input/.../train.csv"
OUTPUT_DIR    = "/kaggle/working/"
VOCAB_GLOSS   = os.path.join(OUTPUT_DIR, "vocab_gloss.json")
VOCAB_TEXT    = os.path.join(OUTPUT_DIR, "vocab_text.json")
MAX_GLOSS_LEN = 12   # max gloss tokens + BOS + EOS
MAX_TEXT_LEN  = 12   # max sentence words + BOS + EOS
VAL_RATIO     = 0.1

# Special tokens — متغيروش! محتاجهم كل الفريق
PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3
# ====================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
df = pd.read_csv(CSV_PATH)
print(f"✅ Loaded {len(df)} samples")
print(df[['sample_id','gloss','sentence']].head(5))


## 🔤 Step 3 — Arabic Text Normalization

العربية محتاجة خطوات تنظيف معينة قبل الـ tokenization.


In [ ]:
def normalize_arabic(text):
    """
    Normalize Arabic text for consistent tokenization.
    Steps:
    1. Remove diacritics (tashkeel: harakat)
    2. Normalize alef variants → ا
    3. Normalize teh marbuta
    4. Remove tatweel (elongation character ـ)
    5. Remove extra whitespace
    """
    pass

# ---- Test ----
examples = [
    "هُوَ مُدَرِّسٌ",        # with diacritics
    "أنا أعرف لغة الإشارة",  # alef variants
    "هـو زميلـي",             # tatweel
]
for ex in examples:
    print(f"Before: {ex}")
    print(f"After:  {normalize_arabic(ex)}")
    print()


## 📊 Step 4 — Dataset Analysis

تحليل الداتاسيت قبل ما تبني الـ vocab.


In [ ]:
# Normalize all gloss and sentence columns
df['gloss_norm']    = df['gloss'].apply(normalize_arabic)
df['sentence_norm'] = df['sentence'].apply(normalize_arabic)

# Gloss statistics
gloss_lengths = df['gloss_norm'].apply(lambda x: len(x.split()))
sent_lengths  = df['sentence_norm'].apply(lambda x: len(x.split()))

print("=" * 45)
print("📊 GLOSS STATISTICS")
print(f"  Avg length:  {gloss_lengths.mean():.2f} tokens")
print(f"  Max length:  {gloss_lengths.max()} tokens")
print(f"  Min length:  {gloss_lengths.min()} tokens")

print("\n📊 SENTENCE STATISTICS")
print(f"  Avg length:  {sent_lengths.mean():.2f} words")
print(f"  Max length:  {sent_lengths.max()} words")
print(f"  Min length:  {sent_lengths.min()} words")

# Count unique tokens
all_gloss_tokens = [t for g in df['gloss_norm'] for t in g.split()]
all_text_words   = [w for s in df['sentence_norm'] for w in s.split()]
print(f"\n📊 VOCABULARY SIZES")
print(f"  Unique Gloss tokens:  {len(set(all_gloss_tokens))}")
print(f"  Unique Text words:    {len(set(all_text_words))}")
print(f"  Total Gloss tokens:   {len(all_gloss_tokens)}")
print(f"  Total Text words:     {len(all_text_words)}")


In [ ]:
# Top 20 most frequent gloss tokens
from collections import Counter
gloss_freq = Counter(all_gloss_tokens)
print("Top 20 Gloss tokens:")
for token, count in gloss_freq.most_common(20):
    print(f"  {token:20s} → {count:4d} times")


## 🏗️ Step 5 — Build Vocabularies

In [ ]:
def build_vocab(token_lists, min_freq=1):
    """
    Build a vocabulary from a list of token lists.
    
    Args:
        token_lists: list of lists of strings
        min_freq: minimum frequency to include a token
    Returns:
        vocab: dict {token: index}
    """
    pass

# Build Gloss vocab
gloss_token_lists = [g.split() for g in df['gloss_norm']]
gloss_vocab = build_vocab(gloss_token_lists, min_freq=1)

# Build Text vocab
text_token_lists = [s.split() for s in df['sentence_norm']]
text_vocab = build_vocab(text_token_lists, min_freq=1)

print(f"✅ Gloss vocab size: {len(gloss_vocab)} tokens")
print(f"✅ Text vocab size:  {len(text_vocab)} tokens")
print(f"\nSample Gloss vocab: {dict(list(gloss_vocab.items())[4:10])}")
print(f"Sample Text vocab:  {dict(list(text_vocab.items())[4:10])}")


## 💾 Step 6 — Save Vocabularies

**مهم جداً:** الـ vocab files دي بتتبعت لـ:
- **CV Engineer 2** → `vocab_gloss.json`
- **NLP Engineer 2** → كلاهما
- **Backend Engineer** → كلاهما


In [ ]:
# Save vocabularies


print(f"✅ Saved: {VOCAB_GLOSS}")
print(f"✅ Saved: {VOCAB_TEXT}")
print(f"✅ Saved: idx2gloss.json + idx2text.json")

# Verify
loaded = json.load(open(VOCAB_GLOSS, encoding="utf-8"))
print(f"\nVerification — Gloss vocab size from file: {len(loaded)}")


## 🔢 Step 7 — Tokenization Functions

دي الدوال المشتركة اللي هيستخدمها NLP Eng 2 في الـ training.


In [ ]:
def tokenize(text, vocab, max_len, normalize=True):
    """
    Convert a text string to a padded integer tensor.
    
    Args:
        text    : input string
        vocab   : dict {token: index}
        max_len : fixed output length (including BOS + EOS)
        normalize: whether to apply Arabic normalization
    Returns:
        tensor of shape (max_len,)
    """
    pass


def detokenize(ids, idx2vocab):
    """
    Convert integer IDs back to a text string.
    Stops at EOS and skips PAD, BOS.
    """
    pass


# ---- Test ----
test_gloss = "هو معلم لغه اشاره"
test_sent  = "هو مدرس لغه اشاره"

g_ids = tokenize(test_gloss, gloss_vocab, MAX_GLOSS_LEN)
s_ids = tokenize(test_sent,  text_vocab,  MAX_TEXT_LEN)

print(f"Gloss: '{test_gloss}'")
print(f"  → IDs: {g_ids.tolist()}")
print(f"  → Back: '{detokenize(g_ids.tolist(), idx2gloss)}'")
print()
print(f"Sentence: '{test_sent}'")
print(f"  → IDs: {s_ids.tolist()}")
print(f"  → Back: '{detokenize(s_ids.tolist(), idx2text)}'")


## 🗄️ Step 8 — PyTorch Dataset Class

الـ `IsharaNLP_Dataset` هيستخدمه NLP Eng 2 مباشرة في الـ training.


In [ ]:
class IsharaNLP_Dataset(Dataset):
    """
    Dataset for Stage 2: Gloss → Text translation.
    
    Returns:
        gloss_ids : tensor (MAX_GLOSS_LEN,)   — encoder input
        text_ids  : tensor (MAX_TEXT_LEN,)    — decoder target
    """
    def __init__(self, df, gloss_vocab, text_vocab,
                 max_gloss_len=12, max_text_len=12):
        pass
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        pass


# ---- Test ----
n_val  = int(len(df) * VAL_RATIO)
df_val = df.iloc[:n_val].reset_index(drop=True)
df_tr  = df.iloc[n_val:].reset_index(drop=True)

train_ds = IsharaNLP_Dataset(df_tr, gloss_vocab, text_vocab, MAX_GLOSS_LEN, MAX_TEXT_LEN)
val_ds   = IsharaNLP_Dataset(df_val, gloss_vocab, text_vocab, MAX_GLOSS_LEN, MAX_TEXT_LEN)

g, t = train_ds[0]
print(f"✅ Train size: {len(train_ds)}  |  Val size: {len(val_ds)}")
print(f"   Gloss IDs shape:    {g.shape}")
print(f"   Sentence IDs shape: {t.shape}")
print(f"   Sample gloss IDs:   {g.tolist()}")
print(f"   Sample sentence IDs:{t.tolist()}")


## ⚡ Step 9 — DataLoader

In [ ]:

print(f"✅ Train DataLoader ready!")


## 📋 Step 10 — حفظ الـ Dataset Info

احفظ ملف summary بكل المعلومات المهمة.


In [ ]:

print("✅ Saved dataset_info.json")


## 📋 Summary — ملخص مسؤوليتك

| Task | Status |
|---|---|
| Load + analyze train.csv | ⬜ |
| Implement Arabic normalization | ⬜ |
| Build Gloss vocabulary (353 tokens) | ⬜ |
| Build Text vocabulary (695 words) | ⬜ |
| Save `vocab_gloss.json` + `vocab_text.json` | ⬜ |
| Implement `tokenize()` + `detokenize()` | ⬜ |
| Build `IsharaNLP_Dataset` | ⬜ |
| Test DataLoader output shapes | ⬜ |

**الـ outputs المطلوبة — بعّتهم لـ:**
```
vocab_gloss.json   → CV Eng 2  +  NLP Eng 2  +  Backend Eng
vocab_text.json    → NLP Eng 2  +  Backend Eng
idx2gloss.json     → Backend Eng
idx2text.json      → Backend Eng
dataset_info.json  → الفريق كله
```
> ✉️ ابعت الـ files دي لـ **NLP Engineer 2** + **CV Engineer 2** + **Backend Engineer**.
